## WildChat dialogue-act annotator (sibling of `check_sharechat.ipynb`)

Same 5-act tutoring taxonomy + LLM pedagogical judge, run on **WildChat-1M** instead of ShareChat.

WildChat has **no `topic` column**, so the **pedagogical judge (cell 3.5) does the tutoring/help-seeking
selection** (not a topic filter). WildChat is conversation-level and carries `language` + `toxic`, so we
gate on **English + non-toxic** at load, then judge, then annotate.

Cells: (0) load + sample WildChat, (1) taxonomy, (2) classifier signature, (3) suite + metrics,
(3.5) pedagogical judge, (4) run acts over kept convos, (5) coverage.

In [1]:
import pandas as pd, random, pathlib, dotenv
dotenv.load_dotenv(pathlib.Path("..") / "wildchat_analysis" / ".env")   # HF_TOKEN for the gated dataset
from datasets import load_dataset

# WildChat-1M is ~1M conversations -> stream + shuffle + take a sample (no full download, no to_pandas).
# WildChat is conversation-level (each row already has the full `conversation` list) and carries
# `language` + `toxic`, so we gate on English + non-toxic here. There is NO topic column, so the
# pedagogical LLM judge (cell 3.5) does the tutoring/help-seeking selection instead of a topic filter.
N_SAMPLE = 400
SEED = 0

_ds = load_dataset("allenai/WildChat-1M", split="train", streaming=True).shuffle(seed=SEED, buffer_size=10000)
wc_convos = []
for ex in _ds:
    if ex.get("language") != "English" or ex.get("toxic"):
        continue
    turns = [{"role": t["role"], "content": t["content"]}
             for t in (ex.get("conversation") or []) if isinstance(t.get("content"), str) and t["content"].strip()]
    if not any(t["role"] == "user" for t in turns):
        continue
    wc_convos.append({"conversation_id": ex["conversation_hash"], "conversation": turns,
                      "n_user_turns": sum(t["role"] == "user" for t in turns)})
    if len(wc_convos) >= N_SAMPLE:
        break
print(f"sampled {len(wc_convos)} English non-toxic WildChat conversations")
print(f"  median user turns/convo: {pd.Series([c['n_user_turns'] for c in wc_convos]).median():.0f}")

sampled 400 English non-toxic WildChat conversations
  median user turns/convo: 1


In [2]:
# (1) Config + taxonomy tables (copied from phase-2 dialogue_act_annotation.py)
import concurrent.futures, logging, pathlib
from collections import Counter
from typing import Literal
import dspy, dotenv

# OPENAI / TOGETHER keys live in the wildchat_analysis/.env; load them here.
dotenv.load_dotenv()
dotenv.load_dotenv(pathlib.Path("..") / "wildchat_analysis" / ".env")

_log = logging.getLogger("dialogue_act_annotation")

# Scheme membership (T = Tutor move, S = Student move)
SCHEME = {
    "Think Aloud": "S", "Conversational Acknowledgment": "S",
    "Knowledge Deficit Question": "S", "Misconception": "S",
    "Common Ground Question": "S", "Vague Answer": "S", "Partial Answer": "S",
    "Social Coordination Action": "S", "Metacomment": "S", "Read Aloud": "S",
    "Solution Request": "S",
    "Forced Choice": "T", "Repetition": "T", "Prompt": "T",
}

# In-context examples per move (from Table I; Solution Request added for human<->AI).
EXAMPLES = {
    "Forced Choice": '"Would that be random, uniformed, or clumped?"',
    "Prompt": '"So 200 times one is what?"',
    "Repetition": 'S: "Is it commensalism?" T: "Commensalism."',
    "Common Ground Question": '"Aren\'t they more lined up, like more in order?"',
    "Conversational Acknowledgment": '"Ok." "No sir." "Yes ma\'am."',
    "Knowledge Deficit Question": '"What do you mean by it doesn\'t have a skeleton?"',
    "Metacomment": '"I don\'t know." "Yes, I understand."',
    "Misconception": '"I always used to get diploid and haploid mixed up."',
    "Partial Answer": '"It has to do with the cells."',
    "Read Aloud": '"Question 7: Plot growth pattern."',
    "Social Coordination Action": '"No, I didn\'t hear about that."',
    "Think Aloud": '"500 equals 50 and 50 divided by 500 gives 10."',
    "Vague Answer": '"Because it helps to, umm, you know."',
    "Solution Request": '"What\'s the best move?" "What now?" "Just tell me what to play." "Any advice for the next move?"',
}

TAXONOMY = ""
for k in SCHEME:
    TAXONOMY += "DIALOGUE ACT: " + k + ", EXAMPLES: " + EXAMPLES[k] + "\n"

DialogueAct = Literal[
    'Think Aloud', 'Conversational Acknowledgment', 'Knowledge Deficit Question',
    'Common Ground Question', 'Solution Request']

print(f"{len(SCHEME)} acts loaded")

15:00:57 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
15:00:57 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


14 acts loaded


In [3]:
# (2) Classifier signature.  <<< EDIT THE DOCSTRING BELOW to tune the task framing. >>>
class DialogueActClassifierSignature(dspy.Signature):
    """Given an utterance from a user in a tutoring or teaching conversation with an AI assistant (the user is learning a topic, or working through it with the assistant's help), first split it into clauses, then classify each clause into the dialogue acts that apply per the taxonomy provided. Output all applicable acts.

Disambiguating the question-type acts (these are easily confused — read carefully):
- "Solution Request": an OPEN request for the assistant to explain, teach, or give the answer, with NO specific candidate proposed. E.g. "explain how photosynthesis works", "teach me how to solve these", "what's the answer?", "walk me through it", "just tell me the steps".
- "Common Ground Question": the user proposes their OWN answer, understanding, or approach and asks the assistant to confirm/evaluate it. E.g. "so mitosis makes two identical cells, right?", "is my understanding correct?", "so it's because of gravity?", "does that mean the answer is 42?".
- "Knowledge Deficit Question": asks about a concept, term, definition, or a specific step — or clarifies something the assistant just said — NOT a request for the whole answer. E.g. "what does 'derivative' mean?", "why does that step work?", "what did you mean by that?".
- "Think Aloud": the user narrates their OWN reasoning or works through the problem out loud, rather than asking or reacting. E.g. "so if the cell loses water it should shrink...", "okay, 12 times 8 is 96, then I add 4"."""
    utterance = dspy.InputField(desc="The utterance to be classified.")
    taxonomy = dspy.InputField(desc="Taxonomy of dialogue acts, with example utterances.")
    dialogue_acts: list[DialogueAct] = dspy.OutputField(desc="The list of dialogue acts applicable to this utterance.")

print(DialogueActClassifierSignature.__doc__.splitlines()[0])

Given an utterance from a user in a tutoring or teaching conversation with an AI assistant (the user is learning a topic, or working through it with the assistant's help), first split it into clauses, then classify each clause into the dialogue acts that apply per the taxonomy provided. Output all applicable acts.


In [4]:
# (3) Panel suite + agreement metrics (copied from phase-2 dialogue_act_annotation.py)
class DialogueActSuite(dspy.Module):
    def __init__(self, arbiter=None, callbacks=None, lm_timeout=120, wall_timeout=150):
        super().__init__(callbacks)
        self.wall_timeout = wall_timeout
        specs = {
            "llama": dspy.LM("together_ai/meta-llama/Llama-3.3-70B-Instruct-Turbo", temperature=0.0, max_tokens=2048, timeout=lm_timeout),
            "gpt": dspy.LM("openai/gpt-5.4-mini", max_tokens=2048, timeout=lm_timeout),
            "sonnet": dspy.LM("anthropic/claude-sonnet-5", max_tokens=2048, timeout=lm_timeout),
        }
        self.annotators = {}
        for name, lm in specs.items():
            a = dspy.ChainOfThought(DialogueActClassifierSignature)
            a.set_lm(lm)
            self.annotators[name] = a
        self.arbiter = None
        if arbiter is not None:
            self.arbiter = dspy.ChainOfThought(DialogueActClassifierSignature)
            self.arbiter.set_lm(arbiter)

    def _annotate(self, name, annotator, utterance):
        try:
            acts = annotator(utterance=utterance, taxonomy=TAXONOMY).dialogue_acts
            valid = set(acts)
            return name, [k for k in SCHEME if k in valid]
        except Exception as e:
            _log.warning("dialogue-act annotator %r failed: %s: %s", name, type(e).__name__, e)
            return name, None

    def forward(self, utterance, min_votes=None):
        per_model = {}
        ex = concurrent.futures.ThreadPoolExecutor(max_workers=len(self.annotators))
        fut_to_name = {ex.submit(self._annotate, name, a, utterance): name
                       for name, a in self.annotators.items()}
        try:
            for fut in concurrent.futures.as_completed(fut_to_name, timeout=self.wall_timeout):
                name, acts = fut.result()
                per_model[name] = acts
        except concurrent.futures.TimeoutError:
            pass
        finally:
            for fut, name in fut_to_name.items():
                if name not in per_model:
                    fut.cancel()
                    _log.warning("dialogue-act annotator %r exceeded wall timeout %ss; dropping", name, self.wall_timeout)
                    per_model[name] = None
            ex.shutdown(wait=False)
        valid = [acts for acts in per_model.values() if acts is not None]
        n_valid = len(valid)
        votes = Counter(act for acts in valid for act in acts)
        if min_votes is None:
            min_votes = n_valid // 2 + 1 if n_valid else 1
        consensus = [k for k in SCHEME if votes[k] >= min_votes]
        confidence = {act: votes[act] / n_valid for act in votes} if n_valid else {}
        no_majority = not consensus and bool(votes)
        needs_review = no_majority or n_valid < 2
        final, arbiter_used = consensus, False
        if no_majority:
            if self.arbiter is not None:
                _, arb_acts = self._annotate("arbiter", self.arbiter, utterance)
                if arb_acts is not None:
                    final, arbiter_used = arb_acts, True
                else:
                    final = [k for k in SCHEME if votes[k] >= 1]
            else:
                final = [k for k in SCHEME if votes[k] >= 1]
        return {
            "final": final, "consensus": consensus, "votes": dict(votes),
            "confidence": confidence, "per_model": per_model, "n_valid": n_valid,
            "min_votes": min_votes, "no_majority": no_majority,
            "needs_review": needs_review, "arbiter_used": arbiter_used,
        }


def _complete_cases(records, raters):
    return [{r: set(rec[r]) for r in raters}
            for rec in records if all(rec.get(r) is not None for r in raters)]

def mean_pairwise_jaccard(records, raters=None):
    raters = raters or sorted({r for rec in records for r in rec})
    items = _complete_cases(records, raters)
    if not items or len(raters) < 2:
        return float("nan")
    pair_scores = []
    for it in items:
        for i in range(len(raters)):
            for j in range(i + 1, len(raters)):
                a, b = it[raters[i]], it[raters[j]]
                pair_scores.append(1.0 if not a and not b else len(a & b) / len(a | b))
    return sum(pair_scores) / len(pair_scores)

def fleiss_kappa(records, raters=None):
    raters = raters or sorted({r for rec in records for r in rec})
    items = _complete_cases(records, raters)
    n = len(raters)
    if len(items) < 2 or n < 2:
        return float("nan"), {}
    kappa_by_act = {}
    for act in SCHEME:
        counts = [sum(act in items[i][r] for r in raters) for i in range(len(items))]
        if all(c == 0 for c in counts) or all(c == n for c in counts):
            continue
        P = [(c * c + (n - c) ** 2 - n) / (n * (n - 1)) for c in counts]
        Pbar = sum(P) / len(P)
        p_present = sum(counts) / (len(counts) * n)
        Pe = p_present ** 2 + (1 - p_present) ** 2
        kappa_by_act[act] = 1.0 if Pe == 1 else (Pbar - Pe) / (1 - Pe)
    macro = sum(kappa_by_act.values()) / len(kappa_by_act) if kappa_by_act else float("nan")
    return macro, kappa_by_act

print("DialogueActSuite + metrics ready")

DialogueActSuite + metrics ready


In [6]:
# (3.5) LLM-as-judge filter: keep only GENUINE learner help-seeking conversations.
#       No topic column in WildChat, so this IS the tutoring/pedagogical selection step.
#       <<< EDIT THE JUDGE DOCSTRING to tune inclusion criteria. >>>
import random
from concurrent.futures import ThreadPoolExecutor as _TPE
from tqdm.auto import tqdm as _tqdm

class PedagogicalHelpJudge(dspy.Signature):
    """Decide whether this conversation is a case of a human LEARNER genuinely seeking to learn or understand something from the AI assistant in a pedagogical setting.

Set learner_help_seeking = True when the user is trying to understand a topic, skill, or problem for THEIR OWN learning: they ask for explanations, work through a problem with the assistant, ask follow-up or clarifying questions, or otherwise try to build their own understanding.

Set learner_help_seeking = False when:
- The user is a TEACHER or AUTHOR requesting teaching ARTIFACTS to give to others (lesson plans, worksheets, quizzes, exams, syllabi, rubrics, class activities). This is content production, not the user's own learning.
- It is a one-off factual lookup with no learning intent or engagement.
- The user only wants a final answer dumped, with no intent to understand it.
- It is content generation, roleplay, or off-topic chit-chat rather than learning."""
    conversation = dspy.InputField(desc="The full conversation (list of role/content turns).")
    user_role: Literal["learner", "teacher_or_author", "other"] = dspy.OutputField(desc="Who the user is in this conversation.")
    learner_help_seeking: bool = dspy.OutputField(desc="True only if this is genuine learner pedagogical help-seeking per the criteria.")
    reason: str = dspy.OutputField(desc="One short sentence justifying the decision.")

_judge = dspy.ChainOfThought(PedagogicalHelpJudge)
_judge.set_lm(dspy.LM("openai/gpt-5.5", timeout=120))

JUDGE_N_CANDIDATES = 300
cands = wc_convos[:JUDGE_N_CANDIDATES]          # already shuffled at load

def _conv(c, maxturns=14):
    return c["conversation"][:maxturns]

def _judge_one(c):
    try:
        r = _judge(conversation=_conv(c))
        return {"cid": c["conversation_id"], "role": r.user_role, "keep": bool(r.learner_help_seeking), "reason": r.reason}
    except Exception as e:
        return {"cid": c["conversation_id"], "role": None, "keep": False, "reason": f"ERROR {type(e).__name__}: {e}"}

with _TPE(max_workers=8) as ex:
    judged = list(_tqdm(ex.map(_judge_one, cands), total=len(cands), desc="judge"))

judged_df = pd.DataFrame(judged)
judged_df.to_json("pedagogical_filter_wildchat.json", orient="records", indent=2)
kept_ids = set(judged_df[judged_df["keep"]]["cid"])
kept_convos = [c for c in cands if c["conversation_id"] in kept_ids]

print(f"judged {len(judged_df)} convos; KEPT {len(kept_convos)} ({len(kept_convos)/len(judged_df):.0%}) as genuine learner help-seeking")
print("user_role:", dict(judged_df["role"].value_counts(dropna=False)))
print("\nsample EXCLUDED:")
for r in judged_df[~judged_df["keep"]].head(5).itertuples(): print(f"  - [{r.role}] {r.reason}")
print("sample KEPT:")
for r in judged_df[judged_df["keep"]].head(5).itertuples(): print(f"  + {r.reason}")

judge:   0%|          | 0/300 [00:00<?, ?it/s]

judged 300 convos; KEPT 51 (17%) as genuine learner help-seeking
user_role: {'other': np.int64(223), 'learner': np.int64(65), 'teacher_or_author': np.int64(12)}

sample EXCLUDED:
  - [other] The interaction is creative content generation rather than learner pedagogical help-seeking.
  - [teacher_or_author] The user is asking for quiz-style content to be generated, not seeking an explanation for their own learning.
  - [teacher_or_author] The user is seeking fictional lore generation rather than pedagogical help for their own learning.
  - [other] The request is for a joke rather than pedagogical help.
  - [teacher_or_author] The user is requesting sentence rewriting/editing rather than trying to learn or understand a concept.
sample KEPT:
  + The user is asking explanatory questions to understand biological systems for their own learning.
  + The user is asking explanatory follow-up questions to build their own understanding of an economics concept.
  + The user is seeking an explanati

In [7]:
# (4) Run the tutoring acts over the JUDGED-pedagogical WildChat conversations.
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

N_CONVOS  = 100     # cap: annotate at most this many kept convos
USE_PANEL = False   # False = single gpt-5.4-mini; True = 3-model suite + majority vote

run = kept_convos[:N_CONVOS]
tasks = [(c["conversation_id"], i, t["content"])
         for c in run
         for i, t in enumerate(c["conversation"])
         if t["role"] == "user" and isinstance(t["content"], str) and t["content"].strip()]
_med = pd.Series([c["n_user_turns"] for c in run]).median() if run else 0
print(f"{len(run)} kept convos (median {_med:.0f} user turns) -> {len(tasks)} user turns"
      + ("  [PANEL x3]" if USE_PANEL else "  [single model]"))

if USE_PANEL:
    _suite = DialogueActSuite()
    def _classify(text):
        r = _suite(utterance=text)
        return r["final"], r["per_model"]
else:
    _single = dspy.ChainOfThought(DialogueActClassifierSignature)
    _single.set_lm(dspy.LM("openai/gpt-5.4-mini", temperature=0.0, max_tokens=2048, timeout=120))
    def _classify(text):
        acts = _single(utterance=text, taxonomy=TAXONOMY).dialogue_acts
        return [k for k in SCHEME if k in set(acts)], None

def _run(task):
    cid, idx, text = task
    try:
        acts, pm = _classify(text)
        return {"conversation_id": cid, "turn": idx, "text": text, "acts": acts, "per_model": pm}
    except Exception as e:
        return {"conversation_id": cid, "turn": idx, "text": text, "acts": None, "per_model": None, "error": f"{type(e).__name__}: {e}"}

with ThreadPoolExecutor(max_workers=8) as ex:
    act_rows = list(tqdm(ex.map(_run, tasks), total=len(tasks), desc="acts"))

acts_df = pd.DataFrame(act_rows)
acts_df.to_json("acts_wildchat_pedagogical.json", orient="records", indent=2)
n_err = acts_df["acts"].isna().sum()
print(f"annotated {len(acts_df) - n_err}/{len(acts_df)} user turns across {acts_df['conversation_id'].nunique()} convos ({n_err} errors) -> acts_wildchat_pedagogical.json")
acts_df.head()

51 kept convos (median 2 user turns) -> 147 user turns  [single model]


acts:   0%|          | 0/147 [00:00<?, ?it/s]

annotated 147/147 user turns across 51 convos (0 errors) -> acts_wildchat_pedagogical.json


,conversation_id,turn,text,acts,per_model
0,26025a5e6bc9b7201b77fe96a43c351e,0,How do lungs of terrestrial organisms create a...,[Solution Request],None
1,8867c79f430d87af3985abe12479edde,0,what the difference between intermediate and f...,[Solution Request],None
2,8867c79f430d87af3985abe12479edde,2,how do capital goods like factories and machi...,[Knowledge Deficit Question],None
3,8867c79f430d87af3985abe12479edde,4,how is the distinction between intermediate an...,[Knowledge Deficit Question],None
4,8867c79f430d87af3985abe12479edde,6,are there any other reasons for only counting ...,[Knowledge Deficit Question],None


In [8]:
# (5) Coverage: does the tutoring taxonomy transfer to WildChat pedagogical help-seeking?
ok = acts_df[acts_df["acts"].notna()].copy()
n_turns = len(ok)
n_none = int((ok["acts"].map(len) == 0).sum())
act_counts = Counter(a for acts in ok["acts"] for a in acts)

coverage = (pd.DataFrame(
        [(a, SCHEME[a], act_counts.get(a, 0), act_counts.get(a, 0) / n_turns) for a in SCHEME],
        columns=["act", "move", "turns_with_act", "frac_of_turns"])
    .sort_values("turns_with_act", ascending=False).reset_index(drop=True))

print(f"{n_turns} user turns classified  (WildChat, judged-pedagogical)")
print(f"  {n_none} ({n_none/n_turns:.0%}) got NO act  <- taxonomy residual")
print(f"  tutor moves (T) fired on: {sum(act_counts.get(a,0) for a in SCHEME if SCHEME[a]=='T')} turns")
if "per_model" in ok and ok["per_model"].notna().any():
    recs = [pm for pm in ok["per_model"] if pm is not None]
    macro, by_act = fleiss_kappa(recs)
    print(f"\npanel: mean pairwise Jaccard {mean_pairwise_jaccard(recs):.3f} | Fleiss' kappa (macro) {macro:.3f}")
coverage

147 user turns classified  (WildChat, judged-pedagogical)
  5 (3%) got NO act  <- taxonomy residual
  tutor moves (T) fired on: 0 turns


,act,move,turns_with_act,frac_of_turns
0,Solution Request,S,102,0.693878
1,Knowledge Deficit Question,S,19,0.129252
2,Think Aloud,S,12,0.081633
3,Conversational Acknowledgment,S,10,0.068027
4,Common Ground Question,S,2,0.013605
5,Misconception,S,0,0.000000
6,Vague Answer,S,0,0.000000
7,Partial Answer,S,0,0.000000
8,Social Coordination Action,S,0,0.000000
9,Metacomment,S,0,0.000000
